<a href="https://colab.research.google.com/github/Alaa-Boghdady/Chest-Xray-Pneumonia-Detection/blob/main/Chest_X_Ray_(Pneumonia_Machine_Learning).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Importing Libs ##

Image
 → Grayscale
 → Resize
 → Normalize
 → Feature Extraction (HOG)
 → Machine Learning Model (SVM)
 → Prediction


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning libs
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from skimage.feature import hog



---



##Data Preprocessing ##

In [ ]:
# Read data and store it in DataFrame
import kagglehub
data = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.


In [ ]:
train_data = '/kaggle/input/chest-xray-pneumonia/chest_xray/train'

filepaths = []
labels = []

folds = os.listdir(train_data)

for fold in folds:
  foldpath = os.path.join(train_data, fold)
  filelist = os.listdir(foldpath)
  for file in filelist:
    fpath = os.path.join(foldpath, file)
    filepaths.append(fpath)
    labels.append(fold)

Fseries = pd.Series(filepaths, name = 'filepaths')
Lseries = pd.Series(labels, name = 'labels')

df = pd.concat([Fseries, Lseries], axis = 1)

In [ ]:
df

,filepaths,labels
0,/kaggle/input/chest-xray-pneumonia/chest_xray/...,PNEUMONIA
1,/kaggle/input/chest-xray-pneumonia/chest_xray/...,PNEUMONIA
2,/kaggle/input/chest-xray-pneumonia/chest_xray/...,PNEUMONIA
3,/kaggle/input/chest-xray-pneumonia/chest_xray/...,PNEUMONIA
4,/kaggle/input/chest-xray-pneumonia/chest_xray/...,PNEUMONIA
...,...,...
5211,/kaggle/input/chest-xray-pneumonia/chest_xray/...,NORMAL
5212,/kaggle/input/chest-xray-pneumonia/chest_xray/...,NORMAL
5213,/kaggle/input/chest-xray-pneumonia/chest_xray/...,NORMAL
5214,/kaggle/input/chest-xray-pneumonia/chest_xray/...,NORMAL


In [ ]:
#Imbalance Data
df['labels'].value_counts()

,count
labels,
PNEUMONIA,3875
NORMAL,1341


In [ ]:
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight(
    class_weight = 'balanced',
    classes = np.unique(labels),
    y = labels
)
class_weights = dict(zip(np.unique(labels), class_weights))



---



##Feature Extraction ##

In [ ]:
features = [] # HOG vector
labels_list = []

#normalise #resize #grayscale
for img_path , label in zip(df['filepaths'], df['labels']):
  #grayscale
  img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
  #resize
  img = cv2.resize(img, (128,128))
  #normalise
  img = img/255

  #HoG
  hog_features = hog(
     img,
     pixels_per_cell = (16,16),
     cells_per_block = (2,2),
     block_norm = 'L2-Hys'
)

  features.append(hog_features)
  labels_list.append(label)



---



##Model ##

In [ ]:
X = np.array(features)
Y = np.array(labels_list)

In [ ]:
X_train, X_temp , Y_train, Y_temp = train_test_split(X, Y, train_size = 0.8, shuffle = True,  random_state = 123)

X_valid, X_test, Y_valid, Y_test = train_test_split(X_temp, Y_temp, train_size = 0.6, shuffle = True,random_state = 123)

print("Train:", "X_train:", X_train.shape,",Y_train:", Y_train.shape)
print("Validation:","X_valid:", X_valid.shape,",Y_valid:", Y_valid.shape)
print("Test:","X_test:" ,X_test.shape,",Y_test:", Y_test.shape)

Train: X_train: (4172, 1764) ,Y_train: (4172,)
Validation: X_valid: (626, 1764) ,Y_valid: (626,)
Test: X_test: (418, 1764) ,Y_test: (418,)


In [ ]:
svm_model = SVC(kernel = 'linear', class_weight = class_weights, probability=True)
log_model = LogisticRegression(max_iter = 1000, class_weight = class_weights)
rf_model  = RandomForestClassifier(n_estimators=100, class_weight = class_weights, random_state = 123)

In [ ]:
svm_model.fit(X_train, Y_train)

SVC(class_weight={np.str_('NORMAL'): np.float64(1.9448173005219984),
                  np.str_('PNEUMONIA'): np.float64(0.6730322580645162)},
    kernel='linear', probability=True)

In [ ]:
log_model.fit(X_train, Y_train)

LogisticRegression(class_weight={np.str_('NORMAL'): np.float64(1.9448173005219984),
                                 np.str_('PNEUMONIA'): np.float64(0.6730322580645162)},
                   max_iter=1000)

In [ ]:
rf_model.fit(X_train, Y_train)

RandomForestClassifier(class_weight={np.str_('NORMAL'): np.float64(1.9448173005219984),
                                     np.str_('PNEUMONIA'): np.float64(0.6730322580645162)},
                       random_state=123)

In [ ]:
treatment_plan = {
    "NORMAL": "No treatment needed. Maintain healthy lifestyle and regular check-ups.",
    "PNEUMONIA": "Consult a doctor immediately. Treatment may include antibiotics, rest, and fluids. Follow medical advice."
}



In [ ]:
def evaluate_model (model, X_test, Y_test):
  y_pred = model.predict(X_test)
  accuracy = accuracy_score(Y_test, y_pred)
  print("Accuracy:", accuracy)
  print("Classification Report:\n", classification_report(Y_test, y_pred))
  print("Confusion Matrix:\n", confusion_matrix(Y_test, y_pred))

In [ ]:
print("=== SVM Model ===")
evaluate_model(svm_model, X_test, Y_test)

print("=== Logistic Regression Model ===")
evaluate_model(log_model, X_test, Y_test)

print("=== Random Forest Model ===")
evaluate_model(rf_model, X_test, Y_test)


=== SVM Model ===
Accuracy: 0.9688995215311005
Classification Report:
               precision    recall  f1-score   support

      NORMAL       0.92      0.95      0.94        99
   PNEUMONIA       0.98      0.97      0.98       319

    accuracy                           0.97       418
   macro avg       0.95      0.96      0.96       418
weighted avg       0.97      0.97      0.97       418

Confusion Matrix:
 [[ 94   5]
 [  8 311]]
=== Logistic Regression Model ===
Accuracy: 0.9617224880382775
Classification Report:
               precision    recall  f1-score   support

      NORMAL       0.88      0.97      0.92        99
   PNEUMONIA       0.99      0.96      0.97       319

    accuracy                           0.96       418
   macro avg       0.94      0.96      0.95       418
weighted avg       0.96      0.96      0.96       418

Confusion Matrix:
 [[ 96   3]
 [ 13 306]]
=== Random Forest Model ===
Accuracy: 0.9521531100478469
Classification Report:
               precision

In [ ]:
hog_features = np.expand_dims(X_test[0], axis=0)
predicted_label = svm_model.predict(hog_features)[0]
plan = treatment_plan[predicted_label]

print("Prediction:", predicted_label)
print("Treatment Plan:", plan)

Prediction: PNEUMONIA
Treatment Plan: Consult a doctor immediately. Treatment may include antibiotics, rest, and fluids. Follow medical advice.
